# Three-Layer Ensemble for Football Match Outcome Prediction

Reproduction of **"A Three-Layer Ensemble Model Combining XGBoost, BiLSTM, and CNN
for Football Match Outcome Prediction"** — Paliwal, Tembhurne & Goud.

| Layer | Component | Paper section |
|---|---|---|
| 1 | XGBoost, BiLSTM, 1D-CNN trained independently | IV-A |
| 2 | Softmax outputs concatenated into `x_meta` ∈ ℝ⁹ | IV-B, Eqs. 12–13 |
| 3 | Multinomial logistic regression meta-learner | IV-C, Eq. 14 |

**Reproduced here:** preprocessing, all three base learners, meta-feature fusion,
LR stacking, Table III (5-fold CV), Table IV (significance tests), Table V
(ablation), Table VI (test performance), and Figs. 1, 2, 4, 5.

> **Read `docs/unspecified_details.md` before interpreting any number.** The
> paper leaves several details unspecified and contradicts itself in two places;
> all 14 such points are documented there rather than silently resolved. The most
> consequential is **#11** — `Match Excitement` and the two team-rating columns
> are post-match quantities that survive the paper's stated drop list, which very
> likely explains the ~88.8% headline accuracy.

**Runtime:** ~5 min for training and evaluation; ~25 min including
cross-validation. CPU is sufficient — the paper itself reports no GPU.

---
## 1. Setup

**If you are running in Colab**, run the cell below to clone the repo. If you
uploaded the folder to your Drive instead, skip it and use the Drive cell.

In [ ]:
# --- Option A: clone from GitHub -------------------------------------------
# !git clone https://github.com/<your-username>/<your-repo>.git
# %cd <your-repo>

# --- Option B: mount Google Drive ------------------------------------------
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/football-ensemble

# --- Option C: running locally ---------------------------------------------
# Nothing to do -- the bootstrap below handles it.

### Locate the project root

This cell walks up from the current directory until it finds `src/config.py`,
then makes that the working directory. It means the notebook imports correctly
whether you launched it from the repo root, from inside `notebooks/`, or from
Colab after a clone — no manual `%cd` needed.

In [ ]:
import os, sys
from pathlib import Path

def find_project_root(start=None, marker="src/config.py", max_up=4):
    here = Path(start or os.getcwd()).resolve()
    for candidate in [here, *here.parents][:max_up + 1]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(
        f"Could not find {marker} in {here} or its parents.\n"
        "Set the working directory to the project root manually."
    )

ROOT = find_project_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Project root:", ROOT)
print("Contents    :", sorted(p.name for p in ROOT.iterdir() if not p.name.startswith('.')))

In [ ]:
# Colab ships with most of these. Uncomment on a fresh environment.
# !pip install -q -r requirements.txt

import sys
print("Python:", sys.version.split()[0])

import numpy, pandas, sklearn, xgboost, tensorflow as tf
print("numpy       :", numpy.__version__)
print("pandas      :", pandas.__version__)
print("scikit-learn:", sklearn.__version__)
print("xgboost     :", xgboost.__version__)
print("tensorflow  :", tf.__version__)

### Dataset

The paper names its input file explicitly:

> *"The file containing the league data to obtain a unified version called
> `combined_data.csv`, where the individual league data are combined."*

So `data/raw/combined_data.csv` is the file used — it is already the merged
five-league file, and no per-league merging step is needed. It ships with this
repository, so the cell below only needs changing if you keep data elsewhere.

In [ ]:
from pathlib import Path
from src import config

print("Expected path:", config.RAW_DATA_FILE)
print("Found        :", config.RAW_DATA_FILE.exists())

if not config.RAW_DATA_FILE.exists():
    # from google.colab import files
    # up = files.upload()
    # Path('data/raw').mkdir(parents=True, exist_ok=True)
    # Path('data/raw/combined_data.csv').write_bytes(up['combined_data.csv'])
    raise FileNotFoundError("Place combined_data.csv in data/raw/")

---
## 2. Load the data and build the target

Section III defines the target from the home team's perspective:

- **Win** — home goals > away goals
- **Loss** — home goals < away goals
- **Draw** — equal

Then, *"to avoid data leakage and ensure valid generalization"*, the identifier,
score and goal columns are dropped.

In [ ]:
from src import data_loader
from src.utils.seed import set_global_seed

set_global_seed(config.RANDOM_SEED)

df, feature_columns = data_loader.load_dataset()

In [ ]:
# The features that survive the paper's drop list.
# Note items 1-3: Match Excitement and the two ratings are POST-match values.
# See docs/unspecified_details.md #11.
for i, c in enumerate(feature_columns, 1):
    print(f"{i:2d}. {c}")

In [ ]:
# Columns removed per Section III
import pandas as pd

pd.DataFrame({
    "Column": config.DROP_COLUMNS,
    "Reason": (["identifier"] * len(config.IDENTIFIER_COLUMNS)
               + ["raw match score"] * len(config.SCORE_COLUMNS)
               + ["goal statistic"] * len(config.GOAL_COLUMNS)),
})

---
## 3. Exploratory data analysis (Figs. 1, 2, 4)

Section III-A. The class imbalance shown here is what motivates the stratified
sampling and the CNN class weights later on.

In [ ]:
from src.utils import plots
from IPython.display import Image, display

labels = df[config.TARGET_COLUMN]

p1 = plots.plot_class_distribution_bar(labels)     # Fig. 1
p4 = plots.plot_class_distribution_pie(labels)     # Fig. 4
p2 = plots.plot_correlation_heatmap(df, feature_columns)  # Fig. 2

display(Image(str(p1)), Image(str(p4)))

In [ ]:
import numpy as np

display(Image(str(p2)))

# Section III-A notes "high correlations between some feature groups, such as
# shooting accuracy, possession, and fouls."
corr = df[feature_columns].corr().abs()
mask = np.triu(np.ones(corr.shape, dtype=bool), k=1)
pairs = corr.where(mask).stack().sort_values(ascending=False)

print("Most correlated feature pairs:")
print(pairs.head(10).round(3).to_string())

---
## 4. Preprocessing and the 80:20 stratified split

Section III specifies two separate scalers:

- **StandardScaler** → XGBoost and BiLSTM
- **Min-Max** → CNN sequence construction

and reshaping for each model:

- XGBoost: tabular `(n, d)`
- BiLSTM: `(n, 1, d)` — *"the input window size is one (non-sequential/sample)"*
- CNN: two `(15, d)` tensors per match, one per team's previous 15 fixtures

All three share **the same split indices**, which Layer 2 requires — Eq. 13
concatenates the base models' outputs row by row.

In [ ]:
from src import train

data = train.prepare_data(df, feature_columns)

print()
for k in ["X_train_std", "X_train_lstm", "home_train", "away_train"]:
    print(f"{k:<15} {data[k].shape}")

---
## 5. Layer 1 — train the three base learners

Hyperparameters are Table II verbatim. Class weights are applied to the **CNN
only**, per Section III: *"while training CNN, we computed class weights and used
them in the loss function."*

In [ ]:
from src.models import xgboost_model, bilstm_model, cnn_model

print("XGBoost :", config.XGBOOST_PARAMS)
print("BiLSTM  :", config.BILSTM_PARAMS)
print("1D-CNN  :", config.CNN_PARAMS)

In [ ]:
# Architecture summaries (Fig. 3, Layer 1)
bilstm_model.build_bilstm(data["X_train_lstm"].shape[1:]).summary()

In [ ]:
cnn_model.build_cnn(data["home_train"].shape[1:]).summary()

---
## 6. Layers 2 and 3 — meta-feature fusion and the meta-learner

This runs Algorithm 1 end to end. Expect roughly 3–5 minutes on CPU.

**One thing to be aware of:** Algorithm 1 fits the meta-learner on the base
models' *in-sample* probabilities, so the measured Layer 3 gain is optimistic.
That is what the paper describes and therefore what runs by default. Set
`config.USE_OUT_OF_FOLD_META_FEATURES = True` for the leakage-free variant.
Details in `docs/unspecified_details.md` #5.

In [ ]:
artifacts = train.train_ensemble(data, seed=config.RANDOM_SEED)

In [ ]:
# The meta-feature matrix of Eq. 13
from src.models import meta_learner

print("X_meta shape:", artifacts.X_meta_train.shape)
pd.DataFrame(
    artifacts.X_meta_train[:5],
    columns=meta_learner.meta_feature_names(),
).round(4)

---
## 7. Evaluation on the held-out test set (Table VI, Fig. 5)

In [ ]:
from src import evaluate

evaluation = evaluate.evaluate_ensemble(artifacts)

In [ ]:
display(Image(str(plots.plot_confusion_matrix(evaluation["confusion_matrix"]))))

In [ ]:
# Section V error analysis:
# "misclassifications occurred most commonly between 'Draw' and 'Loss'"
errors = evaluate.error_analysis(evaluation)

In [ ]:
evaluate.save_evaluation(evaluation)
display(Image(str(config.FIGURES_DIR / "model_comparison.png")))

### Meta-learner interpretability

Section IV-D justifies logistic regression partly on the grounds that *"the
learned weights reflect the contribution of each base model to each class."*
Here are those weights.

In [ ]:
evaluation["meta_weights"].round(3)

---
## 8. Ablation study (Table V)

Section V-D: remove one base model at a time and refit Layer 3. The base learners
are reused rather than retrained, so the measured change reflects removing a
model from the ensemble rather than base-model seed variance.

In [ ]:
from src import ablation

ablation_df = ablation.run_ablation(artifacts)
ablation.save_ablation(ablation_df)

---
## 9. 5-fold stratified cross-validation (Table III)

Section V-B. **This is the slow step — roughly 20 minutes on CPU**, since it
retrains all three base learners five times. Skip it if you only need the
headline numbers.

In [ ]:
from src import cross_validation

cv_summary, cv_per_fold = cross_validation.run_cross_validation(
    df, feature_columns, seed=config.RANDOM_SEED
)
cross_validation.save_cross_validation(cv_summary, cv_per_fold)

In [ ]:
display(Image(str(config.FIGURES_DIR / "cross_validation_accuracy.png")))

---
## 10. Statistical significance testing (Table IV)

Section V-C: two-tailed paired t-test and Wilcoxon signed-rank on the 5 fold
accuracies, at α = 0.01.

**A caveat that matters:** with 5 paired observations the exact two-sided
Wilcoxon test cannot produce a p-value below **0.0625**, so the paper's reported
Wilcoxon values (0.0037, 0.0075, 0.0051) are not attainable from a 5-sample exact
test. Both the exact and normal-approximation p-values are printed below. The
paired t-test is the more defensible of the two at this sample size. See
`docs/unspecified_details.md` #10.

In [ ]:
from src import statistical_tests

tests = statistical_tests.paired_tests(cv_per_fold)
statistical_tests.save_statistical_tests(tests)

---
## 11. Comparison against the paper's reported figures

In [ ]:
paper = pd.DataFrame({
    "Paper accuracy (%)": {
        "XGBoost": 84.77, "BiLSTM": 88.16, "CNN": 46.68, "Ensemble (LR)": 88.78,
    },
    "Paper F1 (%)": {
        "XGBoost": 83.74, "BiLSTM": 87.12, "CNN": 43.89, "Ensemble (LR)": 87.71,
    },
})

ours = pd.DataFrame({
    "This run accuracy (%)": {k: v["accuracy"] * 100
                              for k, v in evaluation["results"].items()},
    "This run F1 macro (%)": {k: v["f1_macro"] * 100
                              for k, v in evaluation["results"].items()},
})

comparison = paper.join(ours)
comparison["Accuracy delta"] = (comparison["This run accuracy (%)"]
                                - comparison["Paper accuracy (%)"])
comparison.round(2)

### Sensitivity check — the post-match feature question

`Match Excitement`, `Home Team Rating` and `Away Team Rating` are assigned by the
data provider **after** a match, informed by how it went. The paper's drop list
does not include them, so this reproduction keeps them.

The cell below retrains without them. If accuracy falls sharply, that tells you
how much of the headline number those three columns were carrying. This is a
diagnostic, **not** part of the paper's methodology — the result above is the
reproduction.

In [ ]:
POST_MATCH = ["Match Excitement", "Home Team Rating", "Away Team Rating"]
pre_match_features = [c for c in feature_columns if c not in POST_MATCH]

print(f"Features: {len(feature_columns)} -> {len(pre_match_features)}")

# Uncomment to run (~5 min):
# data_pm = train.prepare_data(df, pre_match_features)
# art_pm = train.train_ensemble(data_pm, seed=config.RANDOM_SEED)
# eval_pm = evaluate.evaluate_ensemble(art_pm)

---
## 12. Save the trained models

In [ ]:
train.save_models(artifacts)

for p in sorted(config.MODELS_DIR.iterdir()):
    print(f"{p.name:<32} {p.stat().st_size / 1024:>9,.1f} KB")

---
## Summary

Every component of the paper is implemented as specified: preprocessing, the
80:20 stratified split, all three base learners with Table II hyperparameters,
softmax meta-feature fusion, logistic-regression stacking, cross-validation,
significance testing, and the ablation study.

Where the paper is silent or self-contradictory, the choice is documented in
`docs/unspecified_details.md` rather than made silently. The three points most
worth your attention:

1. **#11** — post-match rating columns survive the paper's drop list and are the
   most plausible explanation for accuracy well above the 70–81% literature
   baseline.
2. **#5** — Algorithm 1 specifies in-sample meta-features, which inflates the
   apparent benefit of Layer 3.
3. **#2** — Table II and Section III disagree on the BiLSTM sequence length; the
   prose is followed, which means the BiLSTM is effectively non-recurrent.

None of these are errors in this implementation. They are properties of the
method as published, surfaced so you can decide what to do about them.